In [ ]:
# CCX WP32 shard 02 - config cell
TAG = 'wp32'
SHARD_ID = 2
OUT = '/content/ccx_wp32_shard02.csv'
CONFIG_JSON = '{"groups": [{"n": 500, "d": 3, "noise": "gauss", "kind": "null_gauss", "B": 199, "seeds": [400000, 400001, 400002, 400003, 400004, 400005, 400006, 400007, 400008, 400009, 400010, 400011, 400012, 400013, 400014, 400015, 400016, 400017, 400018, 400019, 400020, 400021, 400022, 400023, 400024, 400025, 400026, 400027, 400028, 400029, 400030, 400031, 400032, 400033, 400034, 400035, 400036, 400037, 400038, 400039, 400040, 400041, 400042, 400043, 400044, 400045, 400046, 400047, 400048, 400049, 400050, 400051, 400052, 400053, 400054, 400055, 400056, 400057, 400058, 400059, 400060, 400061, 400062, 400063, 400064, 400065, 400066, 400067, 400068, 400069, 400070, 400071, 400072, 400073, 400074, 400075, 400076, 400077, 400078, 400079, 400080, 400081, 400082, 400083, 400084, 400085, 400086, 400087, 400088, 400089, 400090, 400091, 400092, 400093, 400094, 400095, 400096, 400097, 400098, 400099]}, {"n": 500, "d": 3, "noise": "gauss", "kind": "null_nonparam", "B": 199, "seeds": [400000, 400001, 400002, 400003, 400004, 400005, 400006, 400007, 400008, 400009, 400010, 400011, 400012, 400013, 400014, 400015, 400016, 400017, 400018, 400019, 400020, 400021, 400022, 400023, 400024, 400025, 400026, 400027, 400028, 400029, 400030, 400031, 400032, 400033, 400034, 400035, 400036, 400037, 400038, 400039, 400040, 400041, 400042, 400043, 400044, 400045, 400046, 400047, 400048, 400049, 400050, 400051, 400052, 400053, 400054, 400055, 400056, 400057, 400058, 400059, 400060, 400061, 400062, 400063, 400064, 400065, 400066, 400067, 400068, 400069, 400070, 400071, 400072, 400073, 400074, 400075, 400076, 400077, 400078, 400079, 400080, 400081, 400082, 400083, 400084, 400085, 400086, 400087, 400088, 400089, 400090, 400091, 400092, 400093, 400094, 400095, 400096, 400097, 400098, 400099]}, {"n": 500, "d": 3, "noise": "t3", "kind": "null_gauss", "B": 199, "seeds": [400000, 400001, 400002, 400003, 400004, 400005, 400006, 400007, 400008, 400009, 400010, 400011, 400012, 400013, 400014, 400015, 400016, 400017, 400018, 400019, 400020, 400021, 400022, 400023, 400024, 400025, 400026, 400027, 400028, 400029, 400030, 400031, 400032, 400033, 400034, 400035, 400036, 400037, 400038, 400039, 400040, 400041, 400042, 400043, 400044, 400045, 400046, 400047, 400048, 400049, 400050, 400051, 400052, 400053, 400054, 400055, 400056, 400057, 400058, 400059, 400060, 400061, 400062, 400063, 400064, 400065, 400066, 400067, 400068, 400069, 400070, 400071, 400072, 400073, 400074, 400075, 400076, 400077, 400078, 400079, 400080, 400081, 400082, 400083, 400084, 400085, 400086, 400087, 400088, 400089, 400090, 400091, 400092, 400093, 400094, 400095, 400096, 400097, 400098, 400099]}]}'
import json as _json
CONFIG = _json.loads(CONFIG_JSON)


In [ ]:
import os, sys
REPO = '/content/ccx-src'
GIT_URL = 'https://github.com/hugogobato/ccx-contextual-confounding.git'
GIT_SHA = 'a643aff94fe71ba26b14b04c7f7126514e1f2b7b'
if not os.path.isdir(REPO):
    r = os.system('git clone -q ' + GIT_URL + ' ' + REPO)
    if r != 0:
        try:
            from google.colab import userdata
            tok = userdata.get('CCX_GH_PAT')
            r = os.system('git clone -q https://' + tok +
                          '@github.com/hugogobato/'
                          'ccx-contextual-confounding.git '
                          + REPO)
        except Exception:
            r = 1
    if r != 0:
        raise RuntimeError('clone failed: repo is private - add a Colab Secret named CCX_GH_PAT (GitHub PAT, repo scope) and rerun')
os.system('git -C ' + REPO + ' checkout -q ' + GIT_SHA)
sys.path.insert(0, REPO + '/src')
import numpy as np
import pandas as pd
from continuous_witness import (k1_witness, k1_multiplier_bootstrap,
                                k2_witness, k2_multiplier_bootstrap,
                                hsic_stat, hsic_bootstrap)
from phase3_dgps import sample_null, sample_confounded
from calibration import critical_values
TRIMS = (0.0, 0.01, 0.05)
ALPHA_GRID = [round(0.01 * a, 2) for a in range(1, 21)]
HSIC_CAP = 2500


In [ ]:
# ---- WP 3.2 null-calibration driver ----
from continuous_witness import (k1_witness, k1_multiplier_bootstrap,
                                k2_witness, k2_multiplier_bootstrap, hsic_stat, hsic_bootstrap)
from phase3_dgps import sample_null, sample_confounded
from calibration import critical_values

def bootstrap_all(x, y, B, bmap, trims, seed):
    _, k1d = k1_multiplier_bootstrap(
        x, y, B=B, trim_grid=trims,
        rng=np.random.default_rng(seed + 7000000), bmap=bmap)
    _, k2d = k2_multiplier_bootstrap(
        x, y, B=B, trim_grid=trims,
        rng=np.random.default_rng(seed + 7100000), bmap=bmap)
    hb = hsic_bootstrap(x[:HSIC_CAP], y[:HSIC_CAP], B=B,
                        rng=np.random.default_rng(seed + 7200000))
    return {"k1": k1d, "k2": k2d, "hsic": {0.0: hb}}


RESUME = os.path.exists(OUT)
done_seeds = set()
if RESUME:
    try:
        prev = pd.read_csv(OUT)
        done_seeds = set((r["kind"], r["noise"], r["n"], r["d"], r["seed"])
                         for _, r in prev.iterrows())
        print("resume:", len(done_seeds), "dataset-records found")
    except Exception:
        print("resume: unreadable partial file, starting fresh")
        RESUME = False

rows_all = []
for gi, g in enumerate(CONFIG["groups"]):
    n, d, noise, kind, B = g["n"], g["d"], g["noise"], g["kind"], g["B"]
    if noise == "gauss":
        bmap = dict((tq, (B if tq == 0.01 else min(B, 49)))
                    for tq in TRIMS)
        trims = TRIMS
    else:
        bmap = None
        trims = (0.01,)
    todo = [s for s in g["seeds"]
            if (kind, noise, n, d, s) not in done_seeds]
    print("[group %d/%d] n=%d d=%d %s %s: %d seeds"
          % (gi + 1, len(CONFIG["groups"]), n, d, noise, kind,
             len(todo)), flush=True)
    for j, seed in enumerate(todo):
        rng = np.random.default_rng(seed)
        x, y, _W = sample_null(rng, n, d, noise=noise, kind=kind)
        obs = {"k1": k1_witness(x, y),
               "k2": k2_witness(x, y),
               "hsic": hsic_stat(x[:HSIC_CAP], y[:HSIC_CAP])}
        boot = bootstrap_all(x, y, B, bmap, trims, seed)
        for meth in ("k1", "k2", "hsic"):
            for tq in (trims if meth != "hsic" else (0.0,)):
                if tq not in boot[meth]:
                    continue
                cvs = critical_values(boot[meth][tq], ALPHA_GRID)
                r = {"n": n, "d": d, "noise": noise, "kind": kind,
                     "seed": seed, "method": meth, "trim": tq,
                     "B": len(boot[meth][tq]), "stat_obs": obs[meth]}
                for a in ALPHA_GRID:
                    r["cv_%.2f" % a] = cvs[a]
                rows_all.append(r)
        if (j + 1) % 10 == 0 or (j + 1) == len(todo):
            pd.DataFrame(rows_all).to_csv(OUT, index=False)
            print("  %d/%d seeds, rows=%d" % (j + 1, len(todo),
                                              len(rows_all)), flush=True)
pd.DataFrame(rows_all).to_csv(OUT, index=False)

manifest = {"tag": TAG, "shard_id": SHARD_ID, "git_sha": GIT_SHA,
            "groups": len(CONFIG["groups"]),
            "rows_written": len(rows_all)}
mpath = os.path.join(os.path.dirname(OUT),
                     "ccx_%s_manifest_shard%02d.json"
                     % (TAG, SHARD_ID))
with open(mpath, "w") as fh:
    json.dump(manifest, fh, indent=2)
print("MANIFEST:", json.dumps(manifest))

try:
    from google.colab import files
    files.download(OUT)
    files.download(mpath)
    print("Downloaded:", OUT)
except Exception as e:
    print("(Not on Colab / download skipped):", e)
